# 06 · Live predictions

Milestone 5 drew win-probability curves for finished matches. This notebook covers games **in progress**:

1. What the API shows about a live game, and how its building masks decode.
2. **What the missing stats cost:** a live game shows only souls, buildings and heroes, so the live model is trained on just those. How much worse is it than the full model?
3. **Live vs reality:** a 2-hour collector run logged 430 ranked games once a minute. After they finished, we scored every logged prediction against the real result.
4. **The clock:** an unexpected finding. The live game time was 3.5 minutes off, and fixing it was the single biggest improvement.

Hidden King = Team0 and Archmother = Team1 in the data. All diffs are Hidden King − Archmother.

In [ ]:
import json
import warnings
from pathlib import Path

import duckdb
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from deadlock_predictor.data.live import PREGAME_S, fetch_active, live_state
from deadlock_predictor.data.match import OBJECTIVES
from deadlock_predictor.features.snapshots import LABEL
from deadlock_predictor.live.score import MATCHES_PATH, clock_check, team_net_worth
from deadlock_predictor.models.data import load_split, predict_saved, snapshot_columns
from deadlock_predictor.models.evaluate import scores, scores_by_time
from deadlock_predictor.viz.curve import TEAM_NAMES, load_models, match_curve, match_features

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS, LIVE = ROOT / "models", ROOT / "data" / "live"
live_train = joblib.load(MODELS / "gbm_live.joblib")  # trained on train only
full_train = joblib.load(MODELS / "gbm_heroes.joblib")  # the M4 final model, train only
live_deploy = joblib.load(MODELS / "gbm_live_deploy.joblib")
deploy = load_models()
plt.rcParams["figure.figsize"] = (9, 4)
warnings.filterwarnings("ignore", message="The inplace keyword")
print("live model inputs:", [c for c in live_train["columns"] if not c.startswith("hero_")])

## 1 · What a live game shows

`/v1/matches/active` lists the games in progress. Per team it gives the current **net worth** and an **objectives mask**; per player the hero; per game the start time. It has no kills, levels, damage, mid-boss or rank: those only exist once the match is processed.

The mask is a 16-bit number: **bit *i* is set while objective *i* still stands** (ids from `data.match.OBJECTIVES`: 1–4 Lane Guardians, 5–8 Walkers, 9 the Patron, 10–11 Shrines, 12–15 Base Guardians). The dump stores the same mask at the end of each match, so we can check the decoding against the per-objective destroyed times:

In [ ]:
games = fetch_active()
example = live_state(games[0])
print(f"{len(games)} ranked games in progress")
pd.Series({k: v for k, v in example.items() if not k.startswith("heroes")})

In [ ]:
# Bit i of a team's mask is set while objective i stands. Check against the dump: at the
# end of a match, an objective with destroyed_time_s == 0 must have its bit set.
NAME_TO_ID = {name: i for i, name in OBJECTIVES.items()}
raw = (ROOT / "data/raw/match_player/partition=123.parquet").as_posix()
rows = duckdb.sql(f"""
    SELECT objectives_mask_team0 AS m0, objectives_mask_team1 AS m1,
           "objectives.team_objective" AS names, "objectives.team" AS owners,
           "objectives.destroyed_time_s" AS destroyed
    FROM '{raw}' WHERE player_slot = 1 AND match_mode = 'Ranked' LIMIT 3000
""").fetchall()
checks = [
    bool((m0 if owner == "Team0" else m1) >> NAME_TO_ID[name] & 1) == (t == 0)
    for m0, m1, names, owners, destroyed in rows
    for name, owner, t in zip(names, owners, destroyed, strict=True)
]
print(f"mask bit agrees with the objective list for {sum(checks):,} of {len(checks):,} objectives")

## 2 · What the missing stats cost

Both models are LightGBM + heroes with the same symmetric training. The only difference is the inputs: the live model gets `t`, `net_worth_diff`, the 5 building diffs and the heroes. Both were trained on **train only**, so the validation split is a fair comparison.

In [ ]:
both = {"full model": full_train, "live model": live_train}
cols = sorted({c for s in both.values() for c in snapshot_columns(s["columns"])})
val = load_split("validation", cols)
y_val = val[LABEL].to_numpy()
val_preds = {name: predict_saved(s, val) for name, s in both.items()}
pd.DataFrame({name: scores(y_val, p) for name, p in val_preds.items()}).T.round(4)

In [ ]:
fig, ax = plt.subplots()
for name, p in val_preds.items():
    st = scores_by_time(val.t.to_numpy(), y_val, p)
    ax.plot(st.index / 60, st.log_loss, marker=".", label=name)
ax.axhline(np.log(2), color="grey", ls="--", label="always 50%")
ax.set(title="Validation log loss by game time", xlabel="minute", ylabel="log loss")
ax.legend();

- **The cost is small:** validation log loss 0.5259 against 0.5183 for the full model. Kills, levels, damage, farm, mid-boss and rank together are worth about 0.008. For scale, adding heroes was worth about 0.03, and the baseline without heroes is 0.5489.
- **Why so small:** most of what kills and farm tell you has already turned into souls. A team that wins fights gets the souls for it, so `net_worth_diff` carries most of that signal.
- **The gap is even across game time.** The two curves are parallel, closest late, when buildings dominate and both models see them.

## 3 · Live vs reality

`python -m deadlock_predictor.live.collect --minutes 120` polled `/active` every minute for 2 hours. Once the games finished, `python -m deadlock_predictor.live.score` fetched each result and scored every logged state, using the **deploy** live model, the one the dashboard uses. Games that fail a training filter (an abandon, no winner) are left out, as in training.

This is the first truly fresh test: games played after every model was trained, with inputs from a different source (the live API, not the post-game data).

In [ ]:
report = json.loads((LIVE / "scores.json").read_text())
scored = pd.read_parquet(LIVE / "scored.parquet")
print(
    json.dumps(
        {
            k: report[k]
            for k in ("logged_games", "finished_games", "scored_games", "scored_states", "left_out")
        },
        indent=1,
    )
)
pd.Series(report["live"]).round(4)

In [ ]:
by_minute = pd.DataFrame(report["by_minute"]).set_index("minute")
rel = pd.DataFrame(report["reliability"]).set_index("bin")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(by_minute.index, by_minute.log_loss, marker="o", label="live, logged games")
st = scores_by_time(val.t.to_numpy(), y_val, val_preds["live model"])
ax1.plot(st.index / 60, st.log_loss, ls=":", label="live model, validation")
ax1.set(title="Log loss by game time", xlabel="minute", ylabel="log loss")
ax1.legend()
ax2.plot([0, 1], [0, 1], color="grey", ls="--")
ax2.plot(rel.predicted, rel.observed, marker="o")
ax2.set(title="Reliability of live predictions", xlabel="predicted", ylabel="observed")
by_minute.round(3)

- **Live log loss 0.503, calibration error 0.024, 74% accuracy** on 9,957 states from 421 games. That's better than the live model's validation 0.526. Two reasons:
  - the scored model is the deploy version, trained on all data up to Oct 25, so its hero strengths are current (the validation number comes from the train-only model, whose hero strengths are weeks old: notebook 04 showed they drift);
  - the live bins are wider: "10" holds minutes 10–14, against single snapshot times in validation, so each live bin sits a bit later in the game, where predictions are easier.
- **Calibration holds:** the reliability points follow the diagonal. When the live model says 75%, the team wins about 75% of the time.
- **Same shape by game time:** hardest at the start (souls are close), easiest around 30–35 minutes, harder again in long games. The 40-minute bin has only 94 states, so it's noisy.
- One evening of games is a small sample (421 games, and states within a game are correlated), so treat the numbers as ±0.01.

## 4 · The clock

The live model needs the game time `t`. The API gives `start_time`, so the first version used `t = now − start_time`. To check it, the scorer compares each logged live team net worth with the post-game net worth snapshots of the same match, interpolated at `t + offset`, for offsets from −300 to +300 s. If the clock is right, the gap is smallest at offset 0.

In [ ]:
finished = pd.read_parquet(MATCHES_PATH)
finished = finished[finished.match_id.isin(scored.match_id)]
post = team_net_worth(finished)
raw = scored.assign(t=(scored.polled_at - scored.start_time).dt.total_seconds())
fig, ax = plt.subplots()
for name, states in {"now − start_time": raw, f"minus PREGAME_S = {PREGAME_S} s": scored}.items():
    clock = clock_check(states, post)
    ax.plot(clock.index, clock.iloc[:, 0], marker=".", label=name)
    gap = clock.iloc[:, 0]
    print(f"{name}: best offset {gap.idxmin():+d} s, gap {gap.min():.3f} (at 0: {gap.loc[0]:.3f})")
ax.set(
    title="Live vs post-game team net worth",
    xlabel="offset added to the live clock (s)",
    ylabel="median relative gap",
)
ax.legend();

- **The raw clock was 3.5 minutes fast.** With `now − start_time`, the gap was smallest at about −225 s: the souls the live API reported matched the post-game souls from ~3.5 minutes *earlier*. At offset 0 the median gap was 25%; at the best offset, 5%.
- **Cause:** `start_time` is set before the game clock starts, which is most likely the pre-game (loading, hero selection). So the live model was told every game was 3.5 minutes older than it was, and at a given souls lead an older game reads as more decided. That made it overconfident.
- **Fix:** `data.live.PREGAME_S = 210`, so `t = now − start_time − 210`. After the fix the best offset is −15 s, one step of the grid, i.e. aligned.
- **Effect:** live log loss went from **0.543 to 0.503**, the biggest single improvement in this milestone, and it came from data plumbing, not modelling. The offset was measured on the same games it was scored on, so 0.503 is slightly flattering. But it's one constant with a physical meaning, not something fitted to the outcomes.

## 5 · Live curves next to post-game curves

For the three logged games with the most polls: the live model once a minute (red), and the post-game curve from the full model at each stat snapshot (black, as in notebook 05).

In [ ]:
many = scored.groupby("match_id").size().sort_values(ascending=False).index[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, mid in zip(axes, many, strict=True):
    rows_m = finished[finished.match_id == mid]
    snaps, heroes, _ = match_features(rows_m)
    post = match_curve(snaps, heroes, deploy)
    live_m = scored[scored.match_id == mid].sort_values("t")
    ax.plot(live_m.t / 60, live_m.p, color="tab:red", label="live (every minute)")
    ax.plot(post.t / 60, post.p_final, marker="o", color="black", label="post-game curve")
    ax.axhline(0.5, color="grey", lw=0.8)
    winner = TEAM_NAMES[rows_m.winning_team.iloc[0]]
    ax.set(title=f"match {mid}: {winner} won", xlabel="minute", ylim=(0, 1))
axes[0].set_ylabel("P(Hidden King wins)")
axes[0].legend(fontsize=8);

- **They agree on the story.** Both curves rise and fall together. The live curve is noisier, because it moves with every poll and sees only souls and buildings.
- **The live curve reacts earlier and jumps more:** a building falling or a fight's souls show up within a minute, while the post-game curve waits for the next snapshot (up to 5 minutes). That's the trade-off of the two models: the live model updates often but sees less, the post-game model sees more but less often.
- After the clock fix the curves line up in time. Before it, the live curve was shifted 3.5 minutes to the right.

## Summary

- **Live inputs:** net worth, building masks (bit *i* = objective *i* standing, verified on 63,009 objectives) and heroes.
- **Live model:** validation 0.5259 vs 0.5183 for the full model. The stats a live game hides are worth little once souls are known.
- **Live check on 421 fresh games:** log loss 0.503, well calibrated.
- **The clock lesson:** `start_time` includes ~210 s of pre-game. Checking inputs against an independent source (post-game snapshots) found it. Fixing it gained more than any model change.
- **Run it yourself:**
  - dashboard: `streamlit run src/deadlock_predictor/app/dashboard.py`, then **Live games**: click a game or type a match id;
  - log games: `python -m deadlock_predictor.live.collect --minutes 120`;
  - score them once they finish: `python -m deadlock_predictor.live.score`.